# Exploratory data analysis(EDA) on provided Marathos Dataset

### Tasks to check off for this EDA:
- Check schema if you have inferred it or specify an appropriate schema
    - Cell 4
---    
- Find out number of rows
    - Cell 9 - Rows: 7 461 195

- Find out number of columns
    - Cell 9 - Colums: 13
---

- descriptive summary of numerical fields
    - Cell 11
---
- count nulls for each field
    - Cell 13
| Athlete performance | Athlete country | Athlete year of birth | Athlete gender | Athlete age category | Athlete average speed |  
| :--- | :--- | :--- | :--- | :--- | :--- |
| 2826373 | 3 | 588161 | 7 | 584938 | 224 |
---

- How many unique events are there?
    - Cell 15 - Unique events: 26907
| Number of Unique events |  
| :--- |  
| 26907 |
---

- How are ages distributed among the runners?
    - Cell 17
    - Cell 19 - Age distribution graph
---

- Which countries are most represented?
    - Cell 21

- How many runners are represented for each country in this dataset?
    - Cell 21 + 22
---
- Graph over Countries with the most runners(top 15)
    - Cell 24
---

### Ingesting my SSOT Bronze data during Development:
- Trying to ingest my data here threw `[DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES]`-error.
    - Will move on write a utility helper script to standardise column naming
    - See last markdown cell in this notebook for detailed code I tried to use

### Validate that my unity catalog setup works prior to loading dataset before EDA begins

In [0]:
%python
# Min volume path för att se så att datasettet finns
VOLUME_PATH = "/Volumes/marathos/default/raw/"
# File path jag ska använda mig utav i min EDA
FILE_PATH = "/Volumes/marathos/default/raw/TWO_CENTURIES_OF_UM_RACES.csv"
display(spark.sql(f"LIST '{VOLUME_PATH}'"))

In [0]:
%python
df_raw = (spark.read.option("header", True)
      .option("inferSchema", True)
      .csv(FILE_PATH)
)
# printar mitt infered schema och hela datasettet för att kunna se varje rows DataTypes
df_raw.printSchema()
display(df_raw)

### Prerequisites for EDA is completed. Now proper EDA can be done to find insights and discrepancies in dataset.

What is done:
- Setup_unity_catalog.sql works as intended
- Inferred schema with its datatypes works as intended

## Imports needed for my EDA

In [0]:
%python
# Imports needed for my EDA - Will import more if needed
from pyspark.sql.functions import col, sum as _sum, countDistinct
# Imports for plotting graphs on dataset

# I need to aggregate in PySpark/Spark and then convert it to Pandass to plot with Plotly
# Trying to plot 7.4mil rows directly in browser will lead to a crash(tested, tried and crashed.)
import plotly.express as px
import pandas as pd

**Find out number of rows + columns in dataset**
- Rows: 7 461 195
- Colums: 13

In [0]:
%python
num_of_rows = df_raw.count()
num_of_cols = len(df_raw.columns)
print(f"Rows: {num_of_rows}, Columns: {num_of_cols}")

### **Summary of numerical fields**

| summary | Year of event | Event dates | Event name | Event distance/length | Event number of finishers | Athlete performance | Athlete club | Athlete country | Athlete year of birth | Athlete gender | Athlete age category | Athlete average speed | Athlete ID |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **count** | 7461195 | 7461195 | 7461195 | 7461195 | 7461195 | 7461193 | 4634822 | 7461192 | 6873034 | 7461188 | 6876257 | 7460971 | 7461195 |
| **mean** | 2011.9834505062527 | null | null | 71.5 | 1451.9687322205089 | null | Infinity | null | 1969.762073925431 | null | null | 778.8179354286929 | 553626.129874236 |
| **stddev** | 10.008437485434591 | null | null | 0.0 | 3124.7836686345695 | null | NaN | null | 13.12559637269166 | null | null | 2612.3201963970423 | 480161.2841795355 |
| **min** | 1798 | 00.00.1895 | "12-Hour Supermarathon 'Kaluzhskiy-92' (RUS)" | 07:35 | 0 | 0.000 km | !Die Gestiefelten Musk... | ACT | 1193.0 | F | F35 | 0 | 0 |
| **max** | 2022 | 31.12.2022-03.01.2023 | Swansea to Cardiff 40 Mile Road Race (GBR) | None | 20027 | 9d 23:51:20 h | ﾓﾝﾍﾞﾙ | swe | 2021.0 | X | WU23 | 9999.0 | 1641167 |



In [0]:
%python
df_raw.describe().display()

**Null counts for each field in dataset**

ONLY counting columns with `NULL-Values` and how many `nulls` are in each column

| Athlete performance | Athlete country | Athlete year of birth | Athlete gender | Athlete age category | Athlete average speed |  
| :--- | :--- | :--- | :--- | :--- | :--- |
| 2826373 | 3 | 588161 | 7 | 584938 | 224 |






In [0]:
%python
null_counts = df_raw.select(
    [_sum(col(c).isNull().cast("int")).alias(c) for c in df_raw.columns]
)

null_counts.display()

### Number of `UNIQUE` events in dataset

| Number of Unique events |  
| :--- |  
| 26907 |


In [0]:
unique_events = df_raw.select("Event name").distinct().count()

print(f"Number of unique events: {unique_events}")
# 26907 unika events i mitt dataset

### What is the AGE distribution of all runners in dataset?

| summary | Age_at_event |
| :--- | :--- |
| **count** | 6873034 |
| **mean** | 42.3923762053265 |
| **stddev** | 9.988186742381977 |
| **min** | -30.0 |
| **25%** | 35.0 |
| **50%** | 42.0 |
| **75%** | 49.0 |
| **max** | 827.0|

In [0]:
%python
df_age = df_raw.withColumn(
    "Age_at_event", col("Year of event") - col("Athlete year of birth")
)
df_age.select("Age_at_event").summary().display()

### Plottting the age distribution
I want to find the age distribution and visualise it for better understanding

In [0]:
%python

# 1: Aggregera datan med PySpark/Spark först(KISS, hämta endast vad jag behöver)
# Filtrera bort alla extrema värden, dvs -30 och 827 som är min och max values i cell 18
age_distribution_spark = (
    df_age.filter((col("Age_at_event") >= 10) & (col("Age_at_event") <= 100))
    .groupBy("Age_at_event")
    .count()
    .orderBy("Age_at_event")
)

# 2: Konvertera min aggregerade table till Pandas
pdf_age = age_distribution_spark.toPandas()

# 3: Skapa min interaktiva plotly graf
fig = px.bar(
    pdf_age,
    x="Age_at_event",
    y="count",
    title="Age distribution of athletes in the dataset(10 - 100 years)",
    labels={"Age_at_event": "Age at event", "count": "Number of athletes"},
    template="plotly_dark",  # Dark theme
)

fig.show()

### What countries are represented in this dataset and HOW many athletes are running for that Country?

- There are a total of 209 unique countries in this Dataset. All nations are using some kind of odd ISO ALPHA-3 country code. I will need to standardise and figure out what iso code it is using find a correct ISO ALPHA 3 dataset to replace it with full nation names instead of just a 3 letter code.


| Athlete country | Number of Athletes |
| :--- | :--- |
| **USA** | 1 389 960 |
| **FRA** | 1 170 884 |
| **RSA** | 877 630 |
| **JPN** | 603 132 |
| **GER** | 442 056 |
| **GBR** | 344 076 |
| **ITA** | 344 076 |
| **ESP** | 232 311 |
| **CHN** | 219 944 |
| **AUS** | 150 761 |
| **TPE** | 143 805 |
| **SUI** | 126 687 |
| **POL** | 119 860 |
| **CAN** | 114 081 |
| **BEL** | 77 868 |



In [0]:
df_raw.groupBy("Athlete country").count().orderBy(
    col("count").desc()
).withColumnRenamed("count", "Number of Athletes").display()

In [0]:
top_15_countries = (
    df_raw.groupBy("Athlete country")
    .count()
    .orderBy(col("count").desc())
    .limit(15)
    .withColumnRenamed("count", "Number of athletes")
)

display(top_15_countries)

## Graph over top 15 nations with most athletes in dataset

In [0]:
# 1: Aggregera country of number of athletes i Spark 
country_distribution_spark = (
    df_raw.groupBy("Athlete country")
    .count()
    .orderBy(col("count").desc())
    .limit(15)
    .withColumnRenamed("count", "Number of athletes")
)

# 2: Konvertera till Pandas för att plotta med Plotly
pdf_country = country_distribution_spark.toPandas()

# 3: Plotly horizontal bar chart över country distribution med antal atleter
fig = px.bar(
    pdf_country,
    x="Number of athletes",
    y="Athlete country",
    orientation="h",
    title="Number of athletes per country (Top 15 nations)",
    labels={"Athlete country": "Country", "Number of athletes": "Number of athletes"},
    template="plotly_dark"
)

fig.show()

## Ingesting data to BRONZE layer
- Ingesting data to my Bronze layer will be done from EDA notebook. Reason I do it here and now is because:
    - No CLEANING is done in Bronze layer.
    - Bronze layer is my SSOT(Single Source of Truth).
    - Raw Bronze data acting as my SSOT *SHOULD* not be changed and *WILL* not be changed.

- Using mode Overwrite for my Bronze data is fine here. Because:
    - When developing my Pipeline and Bronze layer overwrite is good enough since my bronze data will **NOT** change as it acts as my **SSOT**


- This script under did not work. It threw `[DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES]` error directly. Will move on to write a helper function to **standardise** column naming.
```python
# Definiera namnet på table inne i Unity Catalog
bronze_table_name = "marathos.bronze.raw_marathon_data"

print(f"Starting to write raw data to table: {bronze_table_name}")

# Skriv som en DELTA table i Bronze
# NOTE: INGEN CLEANING HÄR! Bronze == SSOT(Single Source Of Truth)
(df_raw.write
 .format("delta")
 .mode("overwrite") # NOTE: Overwrite funkar bra mitt bronze layer aldrig ska ändras och är min SSOT
 .saveAsTable(bronze_table_name))

print("Ingestion to Bronze is Done. The Data is now secured in Delta format")

```